In [ ]:
import numpy as np
import pandas as pd
import re, csv, sys, os
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import IterableDataset, DataLoader

# Increase CSV field size limit
csv.field_size_limit(sys.maxsize)


FILE_PATH = '/kaggle/input/datasets/varshithmgowda/genius-lyrics-dataset/genius_lyrics_subset.csv'
GENRE_COL = 2  
LYRICS_COL = 5 

# --- DATASET MANAGER ---
def stream_samples(filepath=FILE_PATH, limit=None):
    count = 0
    with open(filepath, "r", newline='', encoding='utf-8') as file:
        reader = csv.reader(file)
        headers = next(reader) 
        for row in reader:
            if limit and count >= limit: break
            if len(row) > max(GENRE_COL, LYRICS_COL):
                yield row
                count += 1

# --- EXTRACT TEXT / SONG ---
def simplify_lyrics(lyrics: str):
    lyrics = re.sub(r" +", " ", lyrics)
    lyrics = re.sub(r"[^\w\n., ]", "", lyrics)
    return lyrics.lower().strip()

def stream_annotated_text(limit=None):
    for sample in stream_samples(limit=limit):
        raw_genre = sample[GENRE_COL]
        raw_lyrics = sample[LYRICS_COL]
        
        if not raw_genre or not raw_lyrics: continue
            
        genre_token = f"<{raw_genre.lower().replace(' ', '_')}>"
        lyrics = simplify_lyrics(raw_lyrics)
        
        yield genre_token, lyrics

In [8]:
!pip install sentencepiece
import sentencepiece as spm

# --- TOKENIZE & FORTIFY (SENTENCEPIECE) ---
class SentencePieceVocab:
    def __init__(self, vocab_size=15000):
        self.vocab_size = vocab_size
        self.sp_model = spm.SentencePieceProcessor()
        # Kaggle requires saving files to /kaggle/working/
        self.model_prefix = "/kaggle/working/lyrics_spm" 
        
        self.genre_tokens = [
            "<pop>", "<rock>", "<rb>", "<misc>", "<country>", "<rap>"
        ]

    def train_and_load(self, data_stream):
        print("Extracting text for SentencePiece...")
        temp_file = "/kaggle/working/spm_training_data.txt"
        
        with open(temp_file, "w", encoding="utf-8") as f:
            for genre, text in data_stream:
                f.write(text + "\n")

        print("Training SentencePiece model...")
        user_symbols = ",".join(self.genre_tokens)
        
        spm.SentencePieceTrainer.train(
            input=temp_file,
            model_prefix=self.model_prefix,
            vocab_size=self.vocab_size,
            model_type="unigram", 
            user_defined_symbols=user_symbols, 
            pad_id=0, unk_id=1, bos_id=2, eos_id=3,
            pad_piece="<PAD>", unk_piece="<UNK>", bos_piece="<SOS>", eos_piece="<EOS>"
        )
        
        self.sp_model.load(f"{self.model_prefix}.model")
        if os.path.exists(temp_file): os.remove(temp_file)
        print(f"SentencePiece Built! Total Subword Tokens: {self.sp_model.get_piece_size()}")

    def encode(self, text):
        return self.sp_model.encode_as_ids(text)

    def decode(self, ids):
        return self.sp_model.decode_ids(ids)

    def get_id(self, token):
        if token == "<PAD>": return self.sp_model.pad_id()
        if token == "<UNK>": return self.sp_model.unk_id()
        if token == "<SOS>": return self.sp_model.bos_id()
        if token == "<EOS>": return self.sp_model.eos_id()
        return self.sp_model.piece_to_id(token)

# Build Vocab
vocabulary = SentencePieceVocab(vocab_size=15000)
vocabulary.train_and_load(stream_annotated_text(limit=15000))

Extracting text for SentencePiece...
Training SentencePiece model...
SentencePiece Built! Total Subword Tokens: 15000


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /kaggle/working/spm_training_data.txt
  input_format: 
  model_prefix: /kaggle/working/lyrics_spm
  model_type: UNIGRAM
  vocab_size: 15000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <pop>
  user_defined_symbols: <rock>
  user_defined_symbols: <rb>
  user_defined_symbols: <misc>
  user_defined_symbols: <country>
  user_defined_symbols: <rap>
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_

In [9]:
# --- SEQUENCE PACKING ---
class SlidingWindowDataset(IterableDataset):
    def __init__(self, vocabulary, data_generator, seq_len=15, limit=None):
        self.vocabulary = vocabulary
        self.data_generator = data_generator
        self.seq_len = seq_len
        self.limit = limit

    def __iter__(self):
        for genre_token, song_text in self.data_generator(limit=self.limit):
            encoded_ann = [self.vocabulary.get_id(genre_token)]
            encoded_song = [self.vocabulary.get_id("<SOS>")] + \
                           self.vocabulary.encode(song_text) + \
                           [self.vocabulary.get_id("<EOS>")]

            if len(encoded_song) <= self.seq_len: continue
            
            for i in range(len(encoded_song) - self.seq_len):
                window_x = encoded_song[i : i + self.seq_len]
                target_y = encoded_song[i + self.seq_len]
                yield torch.tensor(encoded_ann), torch.tensor(window_x), torch.tensor(target_y)

def collate_seq2seq(batch):
    anns, windows_x, ys = zip(*batch)
    
    max_ann_len = max(len(a) for a in anns)
    padded_anns = torch.zeros(len(anns), max_ann_len, dtype=torch.long)
    for i, a in enumerate(anns): padded_anns[i, :len(a)] = a
        
    windows_x = torch.stack(windows_x)
    ys = torch.stack(ys)
    
    return padded_anns, windows_x, ys

In [10]:
# --- CONDITIONING METHOD & MODEL ARCHITECTURE ---
class EncoderDecoderLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.encoder_lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.decoder_lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, annotations_x, decoder_x):
        enc_embedded = self.embedding(annotations_x)
        _, (hidden, cell) = self.encoder_lstm(enc_embedded)
        
        dec_embedded = self.embedding(decoder_x)
        outputs, _ = self.decoder_lstm(dec_embedded, (hidden, cell))
        
        last_hidden = outputs[:, -1, :] 
        logits = self.fc(last_hidden)
        
        return logits

In [11]:
import time

# --- FLOW ARCHITECTURE: TRAINING ---
def train_model(model, vocab, epochs=3, batch_size=256, train_limit=None, start_epoch=0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Training on: {device}")

    dataset = SlidingWindowDataset(vocab, stream_annotated_text, seq_len=15, limit=train_limit)
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_seq2seq)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=0) 

    model.train()
    
    for epoch in range(start_epoch, start_epoch + epochs):
        total_loss = 0
        batch_count = 0
        total_data_time = 0
        total_train_time = 0

        data_start = time.time()

        for anns, dec_xs, ys in loader:
            data_time = time.time() - data_start
            total_data_time += data_time

            train_start = time.time()
            anns, dec_xs, ys = anns.to(device), dec_xs.to(device), ys.to(device)

            optimizer.zero_grad()
            logits = model(anns, dec_xs)
            loss = criterion(logits, ys)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_time = time.time() - train_start
            total_train_time += train_time

            total_loss += loss.item()
            batch_count += 1
            
            if batch_count % 500 == 0:
                print(f"Epoch {epoch+1} | Batch {batch_count} | Loss: {loss.item():.4f}")
                print(f"   -> Avg Data Load Time: {total_data_time/batch_count:.4f}s | Avg GPU Train Time: {total_train_time/batch_count:.4f}s")

            data_start = time.time()

        if batch_count > 0:
            avg_loss = total_loss/batch_count
            print(f"=== Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.4f} ===")
            
            # --- CHECKPOINT SAVING (KAGGLE PATH) ---
            save_path = f"/kaggle/working/lyrics_model_epoch_{epoch+1}.pth"
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")
        else:
            print("No batches processed. Check your dataset limit or parsing logic.")


# --- INITIALIZATION & RESUME LOGIC ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = vocabulary.sp_model.get_piece_size()
model = EncoderDecoderLSTM(vocab_size)

# If you ever get disconnected, update this path to point to your saved epoch
RESUME_CHECKPOINT = None 
START_EPOCH = 0 

if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    model.load_state_dict(torch.load(RESUME_CHECKPOINT, map_location=device))
    print(f"Successfully loaded checkpoint: {RESUME_CHECKPOINT}")
    START_EPOCH = int(RESUME_CHECKPOINT.split("_")[-1].split(".")[0]) 

# Unleash the full run!
train_model(model, vocabulary, epochs=3, batch_size=256, train_limit=None, start_epoch=START_EPOCH)

Training on: cuda
Epoch 1 | Batch 500 | Loss: 7.6370
   -> Avg Data Load Time: 0.0052s | Avg GPU Train Time: 0.0069s
Epoch 1 | Batch 1000 | Loss: 7.3606
   -> Avg Data Load Time: 0.0052s | Avg GPU Train Time: 0.0061s
Epoch 1 | Batch 1500 | Loss: 6.0674
   -> Avg Data Load Time: 0.0053s | Avg GPU Train Time: 0.0058s
Epoch 1 | Batch 2000 | Loss: 6.0600
   -> Avg Data Load Time: 0.0053s | Avg GPU Train Time: 0.0057s
Epoch 1 | Batch 2500 | Loss: 5.7316
   -> Avg Data Load Time: 0.0053s | Avg GPU Train Time: 0.0056s
Epoch 1 | Batch 3000 | Loss: 7.0635
   -> Avg Data Load Time: 0.0053s | Avg GPU Train Time: 0.0055s
Epoch 1 | Batch 3500 | Loss: 5.3549
   -> Avg Data Load Time: 0.0053s | Avg GPU Train Time: 0.0055s
Epoch 1 | Batch 4000 | Loss: 6.0205
   -> Avg Data Load Time: 0.0053s | Avg GPU Train Time: 0.0055s
Epoch 1 | Batch 4500 | Loss: 4.8880
   -> Avg Data Load Time: 0.0053s | Avg GPU Train Time: 0.0055s
Epoch 1 | Batch 5000 | Loss: 6.3477
   -> Avg Data Load Time: 0.0053s | Avg GPU Tra

KeyboardInterrupt: 